# Training Model MLP BISINDO (Tahap 3 & 4)

Notebook ini melakukan:
1. Memuat dataset landmark (`data/landmarks_bisindo.csv`)
2. Encoding label kelas dan split dataset (Train / Validation / Test) secara stratified
3. Membangun model MLP sesuai arsitektur pada `ARCHITECTURE.md`
4. Training model dengan `EarlyStopping` dan `ModelCheckpoint`
5. Evaluasi performa model pada test set (Akurasi, Classification Report, Confusion Matrix)
6. Menyimpan model terbaik dan metadata kelas untuk tahap konversi ke TF.js

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## 1. Load Dataset Landmark

In [ ]:
CSV_PATH = '../data/landmarks_bisindo.csv' if os.path.exists('../data/landmarks_bisindo.csv') else 'data/landmarks_bisindo.csv'
df = pd.read_csv(CSV_PATH, dtype={'label': str})
df['label'] = df['label'].astype(str)

print(f"Dataset shape: {df.shape}")
print(f"Jumlah kelas: {df['label'].nunique()}")
print(f"Daftar kelas: {sorted(df['label'].unique())}")
df.head()

## 2. Preprocessing & Stratified Dataset Splitting

- **Train Set**: 70%
- **Validation Set**: 15%
- **Test Set**: 15%

In [ ]:
# Pisahkan fitur (63 koordinat) dan target (label)
X = df.drop(columns=['label']).values.astype(np.float32)
y_raw = df['label'].values

# Encode label ke integer numerik
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
classes = [str(c) for c in label_encoder.classes_]
num_classes = len(classes)

print(f"Jumlah kelas: {num_classes}")
print(f"Classes mapping:")
for idx, cls_name in enumerate(classes):
    print(f"{idx:2d} -> {cls_name}", end="  |  " if (idx + 1) % 6 != 0 else "\n")
print()

# Simpan daftar kelas ke JSON
os.makedirs('../training', exist_ok=True)
os.makedirs('training', exist_ok=True)
classes_json_path = 'training/classes.json' if os.path.exists('training') else '../training/classes.json'
with open(classes_json_path, 'w') as f:
    json.dump(classes, f, indent=2)
print(f"Classes saved to {classes_json_path}")

def stratified_split_dataset(X, y, test_size=0.15, val_size=0.15, random_state=42):
    rng = np.random.RandomState(random_state)
    train_idx, val_idx, test_idx = [], [], []
    classes = np.unique(y)
    for cls in classes:
        cls_indices = np.where(y == cls)[0]
        rng.shuffle(cls_indices)
        n = len(cls_indices)
        if n >= 6:
            n_test = max(1, int(np.round(n * test_size)))
            n_val = max(1, int(np.round(n * val_size)))
            test_idx.extend(cls_indices[:n_test])
            val_idx.extend(cls_indices[n_test : n_test + n_val])
            train_idx.extend(cls_indices[n_test + n_val :])
        elif n >= 3:
            test_idx.extend(cls_indices[:1])
            val_idx.extend(cls_indices[1:2])
            train_idx.extend(cls_indices[2:])
        else:
            train_idx.extend(cls_indices)
    return X[train_idx], X[val_idx], X[test_idx], y[train_idx], y[val_idx], y[test_idx]

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split_dataset(
    X, y_encoded, test_size=0.15, val_size=0.15, random_state=42
)

print(f"\nUkuran Split Dataset:")
print(f"Train set      : {X_train.shape[0]} sample ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set : {X_val.shape[0]} sample ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set       : {X_test.shape[0]} sample ({X_test.shape[0]/len(X)*100:.1f}%)")

## 3. Membangun Arsitektur MLP Model

Sesuai spesifikasi `ARCHITECTURE.md`:
```
Input(63) -> Dense(128, relu) -> Dropout(0.3) -> Dense(64, relu) -> Dropout(0.3) -> Dense(num_classes, softmax)
```

In [ ]:
def create_mlp_model(input_dim: int = 63, num_classes: int = 35) -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(input_dim,), name="landmark_input"),
        layers.Dense(128, activation='relu', name="dense_1"),
        layers.Dropout(0.3, name="dropout_1"),
        layers.Dense(64, activation='relu', name="dense_2"),
        layers.Dropout(0.3, name="dropout_2"),
        layers.Dense(num_classes, activation='softmax', name="classification_output")
    ], name="BISINDO_MLP_Model")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = create_mlp_model(input_dim=63, num_classes=num_classes)
model.summary()

## 4. Training Model dengan Callbacks

In [ ]:
EPOCHS = 100
BATCH_SIZE = 64
MODEL_SAVE_PATH = 'training/model_bisindo.keras' if os.path.exists('training') else '../training/model_bisindo.keras'

training_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-5,
        verbose=1
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=training_callbacks,
    verbose=1
)

## 5. Visualisasi Hasil Training (Loss & Accuracy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='royalblue', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='darkorange', linewidth=2)
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# Loss Plot
axes[1].plot(history.history['loss'], label='Train Loss', color='royalblue', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', color='darkorange', linewidth=2)
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Evaluasi pada Test Set (Tahap 4)

In [ ]:
# Evaluasi langsung dengan model Keras
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"========================================")
print(f"HASIL EVALUASI TEST SET:")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc * 100:.2f}%")
print(f"========================================\n")

# Prediksi kelas untuk test set
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=classes, digits=4, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(16, 13))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=classes,
    yticklabels=classes
)
plt.title('Confusion Matrix — Test Set BISINDO', fontsize=14, pad=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.show()